In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import roc_auc_score
from scipy.stats import uniform, randint

In [2]:
# Datasets
datasets = {
    "heart": ("Data/heart_train.csv", "Data/heart_test.csv"),
    "diabetes": ("Data/diabetes_train.csv", "Data/diabetes_test.csv"),
    "cancer": ("Data/cancer_train.csv", "Data/cancer_test.csv"),
    "alzheimer": ("Data/alzheimer_train.csv", "Data/alzheimer_test.csv")
}

In [3]:
# Grid of hyperparameters 
param_uniform = {
    'max_depth': randint(3, 15),                    # tree depth
    'learning_rate': uniform(0.01, 0.29),           # eta: 0.01-0.3
    'n_estimators': randint(50, 500),               # number of trees
    'subsample': uniform(0.5, 0.5),                 # 0.5-1.0
    'colsample_bytree': uniform(0.5, 0.5),          # 0.5-1.0
    'gamma': uniform(0, 5),                         # min split loss
    'reg_alpha': uniform(0, 1),                     # L1 regularization
    'reg_lambda': uniform(0, 2),                    # L2 regularization
    'min_child_weight': randint(1, 10)              # minimum sum of instance weight
}

In [4]:
all_results = []

for name, (train_path, test_path) in datasets.items():
    print(f"Training: {name}")
    
    # Data load
    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)
    X_train, y_train = train.iloc[:, :-1], train.iloc[:, -1]
    X_test, y_test = test.iloc[:, :-1], test.iloc[:, -1]
    
    # XGBoost model
    xgb = XGBClassifier(
        random_state=42,
        eval_metric='auc'  
    )

    
    # Random Search
    random_search = RandomizedSearchCV(
        estimator=xgb,
        param_distributions=param_uniform,
        n_iter=100,                    
        scoring='roc_auc',
        cv=5,                          
        random_state=42,
        n_jobs=-1
    )
    
    # Fit
    random_search.fit(X_train, y_train)
    
    # Result
    cv_results = pd.DataFrame(random_search.cv_results_)
    
    # Testing on test sets
    for i, params in enumerate(random_search.cv_results_['params']):
        # Training again on parameters from random_search.cv_results_ :(
        model = XGBClassifier(
            random_state=42,
            eval_metric='auc',
            **params
        )
        model.fit(X_train, y_train)
        
        y_proba = model.predict_proba(X_test)[:, 1]
        test_auc = roc_auc_score(y_test, y_proba)
        
        all_results.append({
            "dataset": name,
            "params": params,
            "cv_roc_auc": cv_results.loc[i, 'mean_test_score'],
            "test_roc_auc": test_auc
        })
    
#Results
results_df = pd.DataFrame(all_results)

Training: heart
Training: diabetes
Training: cancer
Training: alzheimer


In [5]:
#Summary
for dataset in datasets.keys():
    dataset_results = results_df[results_df['dataset'] == dataset]
    best_idx = dataset_results['test_roc_auc'].idxmax()
    best_result = dataset_results.loc[best_idx]
    
    print(f"\n{dataset.upper()}:")
    print(f"  Best test AUC: {best_result['test_roc_auc']:.4f}")
    print(f"  CV AUC: {best_result['cv_roc_auc']:.4f}")
    print(f"  Parameters: {best_result['params']}")


HEART:
  Best test AUC: 0.8005
  CV AUC: 0.8026
  Parameters: {'colsample_bytree': 0.6774525952313604, 'gamma': 4.784004425632282, 'learning_rate': 0.20626327228304794, 'max_depth': 6, 'min_child_weight': 3, 'n_estimators': 416, 'reg_alpha': 0.08328441119525964, 'reg_lambda': 0.18340829451696217, 'subsample': 0.8012204629505595}

DIABETES:
  Best test AUC: 0.8411
  CV AUC: 0.8199
  Parameters: {'colsample_bytree': 0.8343216099622155, 'gamma': 4.646879945637929, 'learning_rate': 0.17146123897403964, 'max_depth': 10, 'min_child_weight': 3, 'n_estimators': 222, 'reg_alpha': 0.7694929331919369, 'reg_lambda': 0.37408749711504674, 'subsample': 0.6618396182021218}

CANCER:
  Best test AUC: 0.8721
  CV AUC: 0.8662
  Parameters: {'colsample_bytree': 0.5157145928433671, 'gamma': 3.182052056318902, 'learning_rate': 0.10116323451213473, 'max_depth': 6, 'min_child_weight': 5, 'n_estimators': 456, 'reg_alpha': 0.6044173792778172, 'reg_lambda': 1.0796821826033463, 'subsample': 0.6015306123673847}

A

In [18]:
#results_df.to_csv("Results/test.csv", index=False)

In [21]:
#Finding the best set of hyperparameters for each dataset 
best_per_dataset = (
    results_df.sort_values(by=["dataset", "test_roc_auc"], ascending=[True, False]).groupby("dataset", as_index=False).first()
)
params_df = best_per_dataset["params"].apply(pd.Series)

# Creating new set of hyperparameters from all datasets
mean_params = params_df.mean()

mean_params_dict = mean_params.to_dict()
for param in ["max_depth", "min_child_weight", "n_estimators"]:
    mean_params_dict[param] = int(round(mean_params_dict[param]))
mean_results = []

#Training with new hyperparameters on all datasets
for name, (train_path, test_path) in datasets.items():
    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)

    X_train, y_train = train.iloc[:, :-1], train.iloc[:, -1]
    X_test, y_test = test.iloc[:, :-1], test.iloc[:, -1]

    model = XGBClassifier(
            random_state=42,
            eval_metric='auc',
            **params
        )
    model.fit(X_train, y_train)
    y_proba = model.predict_proba(X_test)[:, 1]
    mean_auc = roc_auc_score(y_test, y_proba)

    mean_results.append({
        "dataset": name,
        "params": params,
        "mean_test_roc_auc": mean_auc
    })
    all_results.append({
            "dataset": name,
            "params": params,
            "cv_roc_auc": cv_results.loc[i, 'mean_test_score'],
            "test_roc_auc": test_auc
        })

mean_df = pd.DataFrame(mean_results)

#Creating new dataframe with all results
results_df = results_df.merge(mean_df, on="dataset")
results_df["diff_from_mean"] = results_df["mean_test_roc_auc"] - results_df["test_roc_auc"]
results_df

MergeError: Passing 'suffixes' which cause duplicate columns {'params_x'} is not allowed.

In [10]:
results_df.to_csv("Results/xgboost_uniform.csv", index=False)